<a href="https://colab.research.google.com/github/KusumaRavuri/Fake_News_Detection_System/blob/main/Fake_News_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required library
!pip install -q nltk

# Import libraries
import pandas as pd
import numpy as np

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load datasets
true_df = pd.read_csv('/content/drive/MyDrive/True.csv')
fake_df = pd.read_csv('/content/drive/MyDrive/Fake.csv')

# Display dataset info
print("True News Shape:", true_df.shape)
print("Fake News Shape:", fake_df.shape)

# NLTK setup
import nltk
nltk.download('stopwords')
nltk.download('punkt')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True News Shape: (21417, 4)
Fake News Shape: (23481, 4)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# Take one real news sample
print(true_df['title'][0])
print(true_df['text'][0][:300])

# Combine them
sample = true_df['title'][0] + " " + true_df['text'][0][:300]

print("\nCombined Sample:\n")
print(sample)

As U.S. budget fight looms, Republicans flip their fiscal script
WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 

Combined Sample:

As U.S. budget fight looms, Republicans flip their fiscal script WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


In [ ]:
import re

# Use already loaded data
fake = fake_df.copy()
real = true_df.copy()

# Labels
fake["label"] = 0
real["label"] = 1

# Add Indian news samples
indian_real = [
    "ISRO successfully launched the Chandrayaan-3 mission to the Moon,marking a major milestone in India's space exploration program, officials said.",

    "According to government reports, India introduced new education policy reforms aimed at improving higher education and skill development across the country.",

    "The Reserve Bank of India reported steady economic growth in the latest quarterly report, highlighting improvements in multiple sectors."
]

indian_fake = [
    "Drinking turmeric water cures all diseases instantly",
    "India bans internet completely starting tomorrow",
    "Aliens spotted in Delhi market shocking everyone"
]

ind_real_df = pd.DataFrame(indian_real, columns=["content"])
ind_real_df["label"] = 1

ind_fake_df = pd.DataFrame(indian_fake, columns=["content"])
ind_fake_df["label"] = 0

# Combine datasets
fake['content'] = fake['title'] + " " + fake['text']
real['content'] = real['title'] + " " + real['text']

data = pd.concat([
    fake[['content','label']],
    real[['content','label']],
    ind_real_df,
    ind_fake_df
])

# Shuffle
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Handle missing safely
data['content'] = data['content'].fillna('')

# Lowercase
data['content'] = data['content'].str.lower()

# Clean text
def clean_text(text):
    text = re.sub(r'[^a-zA-Z ]', '', text)   # remove symbols
    text = re.sub(r'\s+', ' ', text)         # remove extra spaces
    return text.strip()

data['content'] = data['content'].apply(clean_text)

# Final check
print(data.head())
print("Final Shape:", data.shape)

                                             content  label
0  hard core ten gitmo detainees released to oman...      0
1  us energy department balks at trump request fo...      1
2  congress averts government shutdown for now wa...      1
3  vietnam seeks death penalty for embezzlement b...      1
4  trumps spokeswoman flips out on cnn it was a c...      0
Final Shape: (44904, 2)


In [ ]:
from sklearn.model_selection import train_test_split

X = data['content']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check sizes
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (35923,)
Test size: (8981,)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.7,
    min_df=5,
    max_features=5000,
    ngram_range=(1,2),
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# Check shape
print("Train vector shape:", X_train.shape)
print("Test vector shape:", X_test.shape)

Train vector shape: (35923, 5000)
Test vector shape: (8981, 5000)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score

# Predictions
pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9929


In [ ]:
import pickle

# Save model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


STREAMLIT

---



In [ ]:
# Install Streamlit
!pip install -q streamlit

# Install Cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
%%writefile app.py
import streamlit as st
import time
import random
import pickle
import re

# Load model and vectorizer
model = pickle.load(open("model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

# ── Page config ──────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="TruthLens · Fake News Detector",
    page_icon="🔍",
    layout="centered",
    initial_sidebar_state="collapsed",
)

# ── Global CSS ────────────────────────────────────────────────────────────────
st.markdown("""
<style>
/* ---------- imports ---------- */
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Syne:wght@400;600;700;800&display=swap');

/* ---------- reset / base ---------- */
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

html, body, [data-testid="stAppViewContainer"],
[data-testid="stApp"] {
    background: #0a0a0f !important;
    color: #e8e6e1 !important;
    font-family: 'Syne', sans-serif;
}

/* hide streamlit chrome */
#MainMenu, footer, header,
[data-testid="stToolbar"],
[data-testid="stDecoration"]          { display: none !important; }
[data-testid="stSidebar"]             { display: none !important; }

/* main container */
.block-container {
    max-width: 780px !important;
    padding: 1rem 2rem 2rem !important;
}

/* ---------- hero ---------- */
.hero {
    text-align: center;
    padding: 1.5rem 0 1rem;
    position: relative;
}
.hero-badge {
    display: inline-block;
    font-family: 'DM Mono', monospace;
    font-size: 0.65rem;
    letter-spacing: 0.25em;
    text-transform: uppercase;
    color: #5de0a0;
    border: 1px solid #5de0a020;
    background: #5de0a008;
    padding: 0.35rem 0.9rem;
    border-radius: 2px;
    margin-bottom: 1.6rem;
}
.hero h1 {
    font-size: clamp(2.4rem, 6vw, 3.8rem);
    font-weight: 800;
    line-height: 1.05;
    letter-spacing: -0.03em;
    color: #f0ede8;
    margin-bottom: 1rem;
}
.hero h1 span { color: #5de0a0; }
.hero p {
    font-size: 0.95rem;
    color: #6b6b7a;
    font-family: 'DM Mono', monospace;
    letter-spacing: 0.02em;
    max-width: 420px;
    margin: 0 auto;
}

/* ---------- divider ---------- */
.rule {
    height: 1px;
    background: linear-gradient(90deg, transparent, #2a2a3a, transparent);
    margin: 2.5rem 0;
}

/* ---------- label ---------- */
.field-label {
    font-family: 'DM Mono', monospace;
    font-size: 0.7rem;
    letter-spacing: 0.18em;
    text-transform: uppercase;
    color: #5de0a0;
    margin-bottom: 0.6rem;
    display: block;
}

/* ---------- text area ---------- */
textarea {
    background: #111118 !important;
    border: 1px solid #222230 !important;
    border-radius: 4px !important;
    color: #e8e6e1 !important;
    font-family: 'DM Mono', monospace !important;
    font-size: 0.88rem !important;
    line-height: 1.7 !important;
    caret-color: #5de0a0 !important;
    resize: vertical !important;
    transition: border-color 0.2s ease !important;
}
textarea:focus {
    border-color: #5de0a050 !important;
    box-shadow: 0 0 0 3px #5de0a008 !important;
    outline: none !important;
}
textarea::placeholder { color: #33334a !important; }

/* ---------- text input ---------- */
input[type="text"] {
    background: #111118 !important;
    border: 1px solid #222230 !important;
    border-radius: 4px !important;
    color: #e8e6e1 !important;
    font-family: 'DM Mono', monospace !important;
    font-size: 0.88rem !important;
    transition: border-color 0.2s ease !important;
}
input[type="text"]:focus {
    border-color: #5de0a050 !important;
    box-shadow: 0 0 0 3px #5de0a008 !important;
    outline: none !important;
}

/* ---------- button ---------- */
.stButton > button {
    background: #5de0a0 !important;
    color: #0a0a0f !important;
    border: none !important;
    border-radius: 3px !important;
    font-family: 'DM Mono', monospace !important;
    font-size: 0.78rem !important;
    letter-spacing: 0.15em !important;
    text-transform: uppercase !important;
    font-weight: 500 !important;
    padding: 0.75rem 2.2rem !important;
    cursor: pointer !important;
    transition: opacity 0.15s ease, transform 0.15s ease !important;
    width: 100% !important;
}
.stButton > button:hover {
    opacity: 0.88 !important;
    transform: translateY(-1px) !important;
}
.stButton > button:active {
    transform: translateY(0) !important;
}

/* ---------- result cards ---------- */
.verdict-card {
    border-radius: 5px;
    padding: 2rem 2rem 1.6rem;
    margin-top: 2rem;
    position: relative;
    overflow: hidden;
}
.verdict-real {
    background: #0d1f18;
    border: 1px solid #5de0a030;
}
.verdict-fake {
    background: #1f0d0d;
    border: 1px solid #e05d5d30;
}
.verdict-label {
    font-family: 'DM Mono', monospace;
    font-size: 0.65rem;
    letter-spacing: 0.25em;
    text-transform: uppercase;
    margin-bottom: 0.5rem;
}
.label-real    { color: #5de0a0; }
.label-fake    { color: #e05d5d; }

.verdict-title {
    font-size: 2.2rem;
    font-weight: 800;
    letter-spacing: -0.03em;
    line-height: 1;
    margin-bottom: 1.2rem;
}
.title-real    { color: #5de0a0; }
.title-fake    { color: #e05d5d; }

.confidence-row {
    display: flex;
    align-items: center;
    gap: 1rem;
    margin-bottom: 1.2rem;
}
.conf-bar-track {
    flex: 1;
    height: 4px;
    background: #1e1e2a;
    border-radius: 2px;
    overflow: hidden;
}
.conf-bar-fill {
    height: 100%;
    border-radius: 2px;
    transition: width 0.8s cubic-bezier(.16,1,.3,1);
}
.fill-real    { background: #5de0a0; }
.fill-fake    { background: #e05d5d; }

.conf-pct {
    font-family: 'DM Mono', monospace;
    font-size: 0.85rem;
    min-width: 42px;
    text-align: right;
}



/* ---------- stats row ---------- */
.stats-row {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 0.8rem;
    margin: 2.5rem 0 0;
}
.stat-card {
    background: #111118;
    border: 1px solid #1c1c28;
    border-radius: 4px;
    padding: 1.1rem;
    text-align: center;
}
.stat-num {
    font-size: 1.7rem;
    font-weight: 800;
    color: #5de0a0;
    letter-spacing: -0.03em;
    line-height: 1;
    margin-bottom: 0.3rem;
}
.stat-label {
    font-family: 'DM Mono', monospace;
    font-size: 0.65rem;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    color: #44444e;
}

/* ---------- footer ---------- */
.footer {
    text-align: center;
    margin-top: 5rem;
    font-family: 'DM Mono', monospace;
    font-size: 0.68rem;
    letter-spacing: 0.1em;
    color: #2a2a3a;
}

/* ---------- spinner override ---------- */
[data-testid="stSpinner"] > div {
    border-color: #5de0a040 !important;
    border-top-color: #5de0a0 !important;
}

/* ---------- selectbox ---------- */
[data-testid="stSelectbox"] > div > div {
    background: #111118 !important;
    border: 1px solid #222230 !important;
    color: #e8e6e1 !important;
    font-family: 'DM Mono', monospace !important;
    font-size: 0.85rem !important;
    border-radius: 4px !important;
}

/* tabs */
[data-testid="stTabs"] button {
    font-family: 'DM Mono', monospace !important;
    font-size: 0.72rem !important;
    letter-spacing: 0.15em !important;
    text-transform: uppercase !important;
    color: #44444e !important;
}
[data-testid="stTabs"] button[aria-selected="true"] {
    color: #5de0a0 !important;
    border-bottom-color: #5de0a0 !important;
}
</style>
""", unsafe_allow_html=True)


# ── Helpers ───────────────────────────────────────────────────────────────────

def analyze_news(text: str) -> dict:
    # Clean text (same as training)
    cleaned = text.lower()
    cleaned = re.sub(r'[^a-zA-Z ]', '', cleaned)

    # Transform
    vector = vectorizer.transform([cleaned])

    # Predict
    prediction = model.predict(vector)[0]
    prob = model.predict_proba(vector)[0]

    confidence = max(prob) * 100

    if prediction == 1:
        verdict = "REAL"
        summary = "The content appears to be legitimate based on learned patterns."
    else:
        verdict = "FAKE"
        summary = "The content shows patterns similar to misinformation."

    return {
        "verdict": verdict,
        "confidence": confidence,
        "summary": summary,
        "word_count": len(text.split()),
    }

def verdict_classes(v):
    m = {
        "REAL":      ("verdict-real",      "label-real",      "title-real",      "fill-real",      "#5de0a0"),
        "FAKE":      ("verdict-fake",      "label-fake",      "title-fake",      "fill-fake",      "#e05d5d"),
    }
    return m.get(v, m["FAKE"])

# ── UI ────────────────────────────────────────────────────────────────────────

st.markdown("""
<div class="hero">
    <div class="hero-badge">AI-Powered · Real-time Analysis</div>
    <h1>Truth<span>Lens</span></h1>
    <p>Paste any article, headline, or claim. We'll tell you if it checks out.</p>
</div>
""", unsafe_allow_html=True)

st.markdown('<div class="rule"></div>', unsafe_allow_html=True)

# ── Input tabs
tab_text, tab_url = st.tabs(["  PASTE TEXT  ", "  ENTER URL  "])

with tab_text:
    st.markdown('<span class="field-label">Article / Claim</span>', unsafe_allow_html=True)
    user_input = st.text_area(
        label="",
        placeholder="Paste the article text, headline, or claim you want to verify…",
        height=200,
        label_visibility="collapsed",
        key="text_input",
    )

with tab_url:
    st.markdown('<span class="field-label">Article URL</span>', unsafe_allow_html=True)
    url_input = st.text_input(
        label="",
        placeholder="https://example.com/article",
        label_visibility="collapsed",
        key="url_input",
    )
    if url_input:
        user_input = f"[URL submitted: {url_input}] — simulating content fetch for demo purposes."

st.markdown("<br>", unsafe_allow_html=True)

# ── Analyse button
analyze_clicked = st.button("⟶  Analyse Now")

# ── Result
if analyze_clicked:
    text_to_check = user_input.strip() if user_input else ""

    if not text_to_check:
        st.warning("Please paste some text or enter a URL first.")
    else:
        with st.spinner("Running analysis…"):
            result = analyze_news(text_to_check)

        v = result["verdict"]
        conf = result["confidence"]
        card_cls, lbl_cls, title_cls, fill_cls, accent = verdict_classes(v)

        icons = {"REAL": "✓", "FAKE": "✗"}


        st.markdown(f"""
        <div class="verdict-card {card_cls}">
            <div class="verdict-label {lbl_cls}">Verdict</div>
            <div class="verdict-title {title_cls}">{icons[v]} {v}</div>
            <div style="font-family:'DM Mono',monospace;font-size:0.8rem;color:#6b6b7a;margin-bottom:1rem;line-height:1.6;">
                {result["summary"]}
            </div>
            <div class="confidence-row">
                <span class="field-label" style="margin:0;white-space:nowrap;">Confidence</span>
                <div class="conf-bar-track">
                    <div class="conf-bar-fill {fill_cls}" style="width:{conf:.0f}%"></div>
                </div>
                <span class="conf-pct" style="color:{accent}">{conf:.0f}%</span>
            </div>

        </div>
        """, unsafe_allow_html=True)

        # word count note
        st.markdown(f"""
        <div style="font-family:'DM Mono',monospace;font-size:0.68rem;color:#33334a;
                    text-align:right;margin-top:0.6rem;letter-spacing:0.08em;">
            {result["word_count"]} words analysed
        </div>
        """, unsafe_allow_html=True)

Overwriting app.py


In [ ]:
# Run streamlit in background
!streamlit run app.py &> streamlit_logs.txt &

# Start Cloudflare tunnel
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

2026-03-21T09:08:00Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-21T09:08:00Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-21T09:08:03Z INF +--------------------------------------------------------------------------------------------+
2026-03-21T09:08:03Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-21T09:08:03Z INF |  https://rich-cms-numbers-king.trycloudflare.com      